In [1]:
from data_import import load_data, create_train_val_sets
from feature_selection import add_features, drop_features
from model_training import train_and_val_model, ams
import numpy as np
import pandas as pd
import xgboost as xgb

df_train, df_test = load_data()

In [2]:
def create_extended_df(df):
    df = add_features(df)
    df.replace(-999.0, np.nan, inplace = True)
    df.info()
    return df

def create_simplified_df(df):
    df = drop_features(df)
    df.replace(-999.0, np.nan, inplace = True)
    df.info()
    return df

def drop_nans(df):
    df.replace(-999.0, np.nan, inplace = True)
    df.info()
    return df


In [3]:
xgb_model = xgb.XGBClassifier(n_jobs = -1, 
                             tree_method = "hist",
                            )

xgb_param_grid = {"n_estimators": [100, 200],
                  "max_depth": [8, 10],
                  "learning_rate": [0.001,0.005, 0.01],
                }

df_extended = create_extended_df(df_train)
df_simplified = create_simplified_df(df_train)
df_train_nans = drop_nans(df_train)

df_description = ["Starting training with extended features:",
                    "Starting training with simplified features:",
                    "Starting training with original features:"]


<class 'pandas.DataFrame'>
RangeIndex: 654590 entries, 0 to 654589
Data columns (total 36 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   DER_mass_MMC                 554905 non-null  float64
 1   DER_mass_transverse_met_lep  654590 non-null  float64
 2   DER_mass_vis                 654590 non-null  float64
 3   DER_pt_h                     654590 non-null  float64
 4   DER_deltaeta_jet_jet         190277 non-null  float64
 5   DER_mass_jet_jet             190277 non-null  float64
 6   DER_prodeta_jet_jet          190277 non-null  float64
 7   DER_deltar_tau_lep           654590 non-null  float64
 8   DER_pt_tot                   654590 non-null  float64
 9   DER_sum_pt                   654590 non-null  float64
 10  DER_pt_ratio_lep_tau         654590 non-null  float64
 11  DER_met_phi_centrality       654590 non-null  float64
 12  DER_lep_eta_centrality       190277 non-null  float64
 13  PRI_met   

C:\Users\ollea\teknisk-fysik\MLP-final-project\feature_selection.py:30: RuntimeWarning: overflow encountered in sinh
  comp = p * np.sinh(eta)


In [24]:
optimal_models = []

for i, df in enumerate([df_extended, df_simplified, df_train_nans]):
    print(df_description[i])
    optimal_models.append(train_and_val_model(xgb_model, xgb_param_grid, df))

Starting training with extended features:
Training model with param_grid: 
{'n_estimators': [100, 200], 'max_depth': [8, 10], 'learning_rate': [0.001, 0.005, 0.01]} 

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Selected params:
{'learning_rate': 0.005, 'max_depth': 10, 'n_estimators': 100}

Cross val score:
0.886
AMS score: 1.5578
Starting training with simplified features:
Training model with param_grid: 
{'n_estimators': [100, 200], 'max_depth': [8, 10], 'learning_rate': [0.001, 0.005, 0.01]} 

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Selected params:
{'learning_rate': 0.005, 'max_depth': 10, 'n_estimators': 100}

Cross val score:
0.882
AMS score: 1.5363
Starting training with original features:
Training model with param_grid: 
{'n_estimators': [100, 200], 'max_depth': [8, 10], 'learning_rate': [0.001, 0.005, 0.01]} 

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Selected params:
{'learning_rate': 0.005, 'max_depth': 10, 'n_estimato

In [9]:
import joblib

#joblib.dump(optimal_models[0], 'best_extended_model.pkl')
#joblib.dump(optimal_models[1], 'best_simplified_model.pkl')
#joblib.dump(optimal_models[2], 'best_all_features_model.pkl')

['best_all_features_model.pkl']

In [25]:
def test_extended_model(df):
    df = add_features(df)
    df.replace(-999.0, np.nan, inplace = True)
    features = [f for f in df.columns if f not in ["Label", "Weight"]]
    
    X = df[features].to_numpy()
    y = df["Label"].to_numpy()
    weights = df["Weight"].to_numpy()

    y_pred = optimal_models[0].predict(X)
    ams_score = ams(y_pred, y, weights)

    print(f"AMS score for extended features model: {ams_score:.4f}\n")

def test_simplified_model(df):
    df = drop_features(df)
    df.replace(-999.0, np.nan, inplace = True)
    features = [f for f in df.columns if f not in ["Label", "Weight"]]
    
    X = df[features].to_numpy()
    y = df["Label"].to_numpy()
    weights = df["Weight"].to_numpy()

    y_pred = optimal_models[1].predict(X)
    ams_score = ams(y_pred, y, weights)

    print(f"AMS score for simplified model: {ams_score:.4f}\n")

def test_all_features_model(df):
    df.replace(-999.0, np.nan, inplace = True)
    features = [f for f in df.columns if f not in ["Label", "Weight"]]
    
    X = df[features].to_numpy()
    y = df["Label"].to_numpy()
    weights = df["Weight"].to_numpy()

    y_pred = optimal_models[2].predict(X)
    ams_score = ams(y_pred, y, weights)

    print(f"AMS score for original features model: {ams_score:.4f}\n")
    
    
test_extended_model(df_test)
test_simplified_model(df_test)
test_all_features_model(df_test)

C:\Users\ollea\teknisk-fysik\MLP-final-project\feature_selection.py:30: RuntimeWarning: overflow encountered in sinh
  comp = p * np.sinh(eta)


AMS score for extended features model: 1.4170

AMS score for simplified model: 1.4123

AMS score for original features model: 1.4538

